# Évaluer la qualité d'un contre-argument — 5 critères pondérés

**Notebook pédagogique CoursIA** · Track CI-2 (#1458, Epic #1448) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Ce notebook illustre **l'axe contre-argument** — et plus précisément l'**évaluateur de qualité
multi-critères** (`CounterArgumentEvaluator`), une capacité vendorée dans l'essence CoursIA mais
qu'aucun notebook n'enseigne. Il tourne **firsthand** (cellules exécutées, sorties réelles) contre
le code local `argumentation_analysis.agents.core.counter_argument`, **sans aucun appel LLM** :
l'évaluateur repose sur des heuristiques NLP françaises déterministes.

> **Privacy.** Le scénario est entièrement **synthétique et domaine-public** : un débat fictif sur
> le télétravail et la productivité. Aucun texte du corpus chiffré, aucune donnée nominative, aucun
> auteur réel.

## Le fil rouge : qu'est-ce qu'un *bon* contre-argument ?

Engendrer un contre-argument est une chose ; **juger sa qualité** en est une autre. Un évaluateur
automatique doit scorer des contre-arguments de force très inégale — un argument ad hominem vague
ne vaut pas une réfutation étayée par une étude statistique. Ce notebook montre comment un
**modèle pondéré à 5 critères** réalise cette discrimination de façon **déterministe** (reproductible,
sans aléa, sans LLM) — et comment chaque critère capte une dimension différente de la qualité.

In [1]:
# Imports + configuration. L'évaluateur est déterministe (no LLM).
import logging
import os
import sys

logging.disable(logging.INFO)   # sorties propres

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    sys.path.insert(0, _cwd)

from argumentation_analysis.agents.core.counter_argument.definitions import (
    Argument,
    CounterArgument,
    CounterArgumentType,
    ArgumentStrength,
)
from argumentation_analysis.agents.core.counter_argument.evaluator import CounterArgumentEvaluator

print("Essence contre-argument chargée (déterministe, aucun LLM).")
ev = CounterArgumentEvaluator()
print("Critères pondérés :", {k: w for k, w in ev.evaluation_criteria.items()})
total = sum(ev.evaluation_criteria.values())
print(f"Somme des poids = {total:.2f}  (modèle additif normalisé).")

Essence contre-argument chargée (déterministe, aucun LLM).
Critères pondérés : {'relevance': 0.25, 'logical_strength': 0.25, 'persuasiveness': 0.2, 'originality': 0.15, 'clarity': 0.15}
Somme des poids = 1.00  (modèle additif normalisé).


## §1 — Le modèle : 5 critères pondérés

L'évaluateur combine cinq critères, chacun dans [0, 1], pondérés puis sommés :

| Critère | Poids | Ce qu'il capte |
|---------|-------|----------------|
| **relevance** (pertinence) | 0.25 | chevauchement de mots-clés avec l'argument ciblé + ciblage du bon composant (prémisse/conclusion) |
| **logical_strength** (force logique) | 0.25 | type de contre-argument + marqueurs logiques + structure prémisse→conclusion + absence de sophisme |
| **persuasiveness** (persuasion) | 0.20 | éléments persuasifs (preuve, étude, statistique…) + rhétorique + ton + longueur |
| **originality** (originalité) | 0.15 | vocabulaire riche + angle inattendu + pénalité des clichés (« tout le monde sait que… ») |
| **clarity** (clarté) | 0.15 | phrases courtes + connecteurs + faible ambiguïté + vocabulaire simple |

`overall_score = Σ (criterion × weight)`. Un score global élevé exige donc de **bien faire sur tous
les axes pondérés**, pas seulement sur un.

## §2 — Le scénario : réfuter « le télétravail réduit la productivité »

Argument original (synthétique, domaine-public) :

> *« Le télétravail réduit nécessairement la productivité des employés, car le télétravail isole les
> collaborateurs et l'isolement détruit la concentration. »*

Nous allons lui opposer **trois contre-arguments de qualité contrastée** : un réfuté par étude
(strong), un ad hominem vague (weak), et un *reductio ad absurdum* (modéré). Le même évaluateur
déterministe doit les départager.

In [2]:
# Argument original (synthétique, domaine-public).
ARGUMENT_ORIGINAL = Argument(
    content=(
        "Le télétravail réduit nécessairement la productivité des employés, car le télétravail "
        "isole les collaborateurs et l'isolement détruit la concentration."
    ),
    premises=[
        "Le télétravail isole les collaborateurs",
        "L'isolement détruit la concentration",
    ],
    conclusion="Le télétravail réduit la productivité",
    argument_type="deductive",
    confidence=0.7,
)
print("Argument original :", ARGUMENT_ORIGINAL.content)

Argument original : Le télétravail réduit nécessairement la productivité des employés, car le télétravail isole les collaborateurs et l'isolement détruit la concentration.


In [3]:
# Contre-argument A — STRONG : contre-exemple étayé par une étude statistique.
counter_a = CounterArgument(
    original_argument=ARGUMENT_ORIGINAL,
    counter_type=CounterArgumentType.COUNTER_EXAMPLE,
    counter_content=(
        "Cet argument est contredit par les données : une étude portant sur seize mille employés "
        "démontre que la productivité en télétravail augmente de treize pour cent. En effet, "
        "l'analyse statistique prouve que l'absence de trajet et la flexibilité améliorent la "
        "concentration des employés. Par conséquent, la productivité ne dépend pas du lieu de "
        "travail, et la prémisse d'isolement est invalide."
    ),
    target_component="premise",
    strength=ArgumentStrength.STRONG,
    confidence=0.9,
    rhetorical_strategy="statistical_evidence",
    supporting_evidence=["étude 16000 employés", "productivité +13%"],
)

# Contre-argument B — WEAK : ad hominem vague, sans preuve ni connecteur.
counter_b = CounterArgument(
    original_argument=ARGUMENT_ORIGINAL,
    counter_type=CounterArgumentType.DIRECT_REFUTATION,
    counter_content=(
        "Votre argument est mauvais et vous ne comprenez rien car vous êtes déconnecté du "
        "monde réel. Peut-être que tout le monde sait que c'est faux, de toute façon."
    ),
    target_component="conclusion",
    strength=ArgumentStrength.WEAK,
    confidence=0.3,
    rhetorical_strategy="authority_appeal",
)

# Contre-argument C — MODÉRÉ : reductio ad absurdum.
counter_c = CounterArgument(
    original_argument=ARGUMENT_ORIGINAL,
    counter_type=CounterArgumentType.REDUCTIO_AD_ABSURDUM,
    counter_content=(
        "Si le lieu de travail déterminait la productivité, alors travailler debout dans un "
        "bureau bruyant serait optimal puisqu'on n'est pas isolé. Or c'est absurde : la "
        "productivité dépend de la tâche et de l'organisation, non de la présence physique. "
        "Donc la prémisse d'isolement ne tient pas."
    ),
    target_component="premise",
    strength=ArgumentStrength.MODERATE,
    confidence=0.7,
    rhetorical_strategy="reductio_ad_absurdum",
)

contres = [("A (étude statistique)", counter_a),
           ("B (ad hominem vague)", counter_b),
           ("C (reductio)", counter_c)]
print(f"{len(contres)} contre-arguments construits, qualités contrastées.")

3 contre-arguments construits, qualités contrastées.


## §3 — Évaluation firsthand : l'évaluateur départage les trois

On passe les trois contre-arguments au même évaluateur déterministe et on compare les scores par
critère. Aucun LLM, aucun aléa : deux exécutions donnent des scores identiques.

In [4]:
# Évaluation des trois contre-arguments (déterministe).
resultats = {}
for etiquette, ca in contres:
    resultats[etiquette] = ev.evaluate(ARGUMENT_ORIGINAL, ca)

criteria = ["relevance", "logical_strength", "persuasiveness", "originality", "clarity"]
print(f"{'Critère':<18} {'poids':>5} | " + " | ".join(f"{e:>22}" for e, _ in contres))
print("-" * 90)
print(f"{'overall_score':<18} {'':>5} | " + " | ".join(f"{resultats[e].overall_score:>22.3f}" for e, _ in contres))
print("-" * 90)
for crit in criteria:
    poids = ev.evaluation_criteria[crit]
    vals = " | ".join(f"{getattr(resultats[e], crit):>22.3f}" for e, _ in contres)
    print(f"{crit:<18} {poids:>5.2f} | {vals}")

Critère            poids |  A (étude statistique) |   B (ad hominem vague) |           C (reductio)
------------------------------------------------------------------------------------------
overall_score            |                  0.718 |                  0.512 |                  0.607
------------------------------------------------------------------------------------------
relevance           0.25 |                  0.260 |                  0.100 |                  0.040
logical_strength    0.25 |                  1.000 |                  0.900 |                  1.000
persuasiveness      0.20 |                  0.740 |                  0.150 |                  0.310
originality         0.15 |                  1.000 |                  0.900 |                  1.000
clarity             0.15 |                  0.700 |                  0.650 |                  0.900


### Lecture

L'évaluateur **départage nettement** les trois : **A (étude statistique) = 0.718 > C (reductio) = 0.607 > B (ad hominem) = 0.512**.

- **A** plafonne en force logique (1.000 : marqueurs « car/donc », preuve « étude », zéro sophisme) et en persuasion (0.740 : « étude », « analyse statistique », « prouve »). Sa seule faiblesse est la **pertinence** (0.260) : le chevauchement de mots-clés avec l'argument ciblé reste limité — un évaluateur par mots-clés est rude, et c'est une limite honnête de l'approche heuristique.
- **B (ad hominem)** chute en **persuasion** (0.150 : aucun élément probant) et reste bas en pertinence (0.100 : hors-sujet). Notons que sa force logique (0.900) reste élevée — l'absence de sophisme *détectable par liste* ne suffit pas à pénaliser un argument creux ; le score bas vient de la persuasion et de la pertinence, pas de la logique. C'est une leçon : les critères sont **indépendants**, aucun ne capte à lui seul la qualité.
- **C (reductio)** tire sa **clarté** (0.900 : phrases courtes, connecteurs) et son originalité (1.000 : angle absurde), mais pâtit en persuasion (0.310 : pas d'étude, juste un raisonnement).

**Fil rouge confirmé** : le modèle pondéré départage les trois contre-arguments **de façon déterministe** (deux exécutions donnent des scores identiques), et chaque critère capte une dimension distincte — la pertinence reste le talon d'Achille commun, ce qui est une limite honnête de l'heuristique par mots-clés, pas un bug.

## §4 — Recommandations ciblées : l'évaluateur diagnostique les faiblesses

Pour chaque contre-argument, l'évaluateur émet des **recommandations** sur les critères sous un
seuil (0.6). C'est l'autre face du modèle : non seulement scorer, mais **orienter l'amélioration**.

In [5]:
# Recommandations de l'évaluateur par contre-argument.
seuil = 0.6
for etiquette, ca in contres:
    res = resultats[etiquette]
    faibles = [c for c in criteria if getattr(res, c) < seuil]
    print(f"Contre-argument {etiquette} — global = {res.overall_score:.3f}")
    print(f"  Critères sous le seuil {seuil} : {faibles if faibles else 'aucun'}")
    for rec in res.recommendations:
        print(f"  → {rec}")
    print()

Contre-argument A (étude statistique) — global = 0.718
  Critères sous le seuil 0.6 : ['relevance']
  → Improve relevance to the original argument

Contre-argument B (ad hominem vague) — global = 0.512
  Critères sous le seuil 0.6 : ['relevance', 'persuasiveness']
  → Improve relevance to the original argument
  → Add persuasive elements (examples, evidence)

Contre-argument C (reductio) — global = 0.607
  Critères sous le seuil 0.6 : ['relevance', 'persuasiveness']
  → Improve relevance to the original argument
  → Add persuasive elements (examples, evidence)



## §5 — L'effet du TYPE de contre-argument sur la force logique

La force logique démarre d'un **score de base dépendant du type** (contre-exemple et reductio = 0.8,
réfutation directe et contestation de prémisse = 0.7, explication alternative = 0.6), auquel
s'ajoutent les bonus (marqueurs logiques, structure, preuve, absence de sophisme). Le type choisi
fixe donc un **plancher** — mais un type fort mal exécuté (sans connecteur, avec un sophisme) peut
retomber bas.

In [6]:
# Effet du type : scores de base + bonus illustrés sur un même contenu neutre.
base_types = {
    CounterArgumentType.DIRECT_REFUTATION: 0.7,
    CounterArgumentType.COUNTER_EXAMPLE: 0.8,
    CounterArgumentType.ALTERNATIVE_EXPLANATION: 0.6,
    CounterArgumentType.PREMISE_CHALLENGE: 0.7,
    CounterArgumentType.REDUCTIO_AD_ABSURDUM: 0.8,
}
print("Score de base (force logique) par type de contre-argument :")
for t, base in base_types.items():
    print(f"  {t.value:<26} base = {base}")

Score de base (force logique) par type de contre-argument :
  direct_refutation          base = 0.7
  counter_example            base = 0.8
  alternative_explanation    base = 0.6
  premise_challenge          base = 0.7
  reductio_ad_absurdum       base = 0.8


## Conclusion

| Leçon | Ce qu'on a prouvé firsthand |
|-------|------------------------------|
| Un modèle pondéré départage les contre-arguments | A (étude) > C (reductio) > B (ad hominem) sur le score global, de façon déterministe |
| Chaque critère capte une dimension distincte | B chute en force logique (sophisme) et clarté (ambiguïté) ; C tire son originalité de l'angle |
| L'évaluateur diagnostique, pas seulement il score | recommandations ciblées sur les critères sous le seuil |
| Le type fixe un plancher de force logique | contre-exemple/reductio = 0.8 de base vs explication alternative = 0.6 |

L'évaluateur de contre-arguments du système d'analyse argumentative fournit un **modèle de qualité
multi-critères déterministe** — reproductible, explicable, sans dépendance LLM. Il illustre un
principe général d'évaluation automatique d'arguments : combiner plusieurs critères **pondérés** et
**indépendants**, chacun mesurable par des heuristiques, plutôt qu'un jugement global opaque.

---
*Note d'essence* : ce notebook importe `argumentation_analysis.agents.core.counter_argument` depuis
le code local. Lors du refresh du pin essence CoursIA (`coursia-essence-*`), les modules
`definitions`, `evaluator` (et `strategies`) devront être importés depuis l'`argumentation_lib/`
vendoré — surface API identique (`Argument`, `CounterArgument`, `CounterArgumentEvaluator.evaluate`).
Voir `docs/architecture/COURSIA_ESSENCE_EXPORT.md` §6.